# MuseTalk 1.5 GPU validation worker

This notebook is a **test worker**, not a production scheduler. It takes an approved singer image/video and the successful ACE-Step audio, runs MuseTalk 1.5 on a Colab GPU, and saves the lip-synced MP4.

Requirements: enable a GPU runtime in Colab. Do not publish the result until visual QA confirms identity, clothing, devotional scene and lip-sync.

In [ ]:
!nvidia-smi
!git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git
%cd MuseTalk
!pip install -q -r requirements.txt
!pip install -q ffmpeg-python


In [ ]:
# Download model weights using the official project mechanism.
# If the upstream repository changes its weight downloader, follow its current README here.
!bash ./download_weights.sh


In [ ]:
from google.colab import files
print('Upload the APPROVED traditional-clothing singer image/video:')
uploaded_avatar = files.upload()
print('Upload the successful ACE-Step bhajan MP3/WAV:')
uploaded_audio = files.upload()
avatar = next(iter(uploaded_avatar))
audio = next(iter(uploaded_audio))
print('Avatar:', avatar)
print('Audio:', audio)


In [ ]:
import os, subprocess, pathlib
out = pathlib.Path('/content/musetalk_output')
out.mkdir(exist_ok=True)
# Normalize source media before inference.
subprocess.run(['ffmpeg','-y','-i',avatar,'-vf','scale=512:-2','-pix_fmt','yuv420p',str(out/'avatar.mp4')],check=True) if avatar.lower().endswith(('.mp4','.mov','.webm')) else subprocess.run(['ffmpeg','-y','-loop','1','-i',avatar,'-t','5','-vf','scale=512:-2','-pix_fmt','yuv420p',str(out/'avatar.mp4')],check=True)
subprocess.run(['ffmpeg','-y','-i',audio,'-ar','16000','-ac','1',str(out/'audio.wav')],check=True)


In [ ]:
# MuseTalk inference.
# Keep this command aligned with the upstream README if CLI arguments change.
!python -m scripts.inference --inference_config configs/inference/test.yaml --result_dir /content/musetalk_output/result --unet_model_path models/musetalkV15/unet.pth --whisper_dir models/whisper --version v15


In [ ]:
from pathlib import Path
from google.colab import files
candidates=list(Path('/content/musetalk_output/result').rglob('*.mp4'))
assert candidates, 'MuseTalk produced no MP4'
candidate=max(candidates,key=lambda p:p.stat().st_size)
assert candidate.stat().st_size > 100_000
print('SUCCESS:',candidate)
files.download(str(candidate))
